In [55]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"C:\Users\USER\OneDrive\Desktop\Business_Analysis_Agent\data\processed\zomato_cleaned.csv")
df.shape

(51696, 10)

In [56]:
df_fe = df.copy()

print("Feature engineering dataset shape:", df_fe.shape)

Feature engineering dataset shape: (51696, 10)


In [57]:
df_fe["online_order"] = (
    df_fe["online_order"]
    .map({"Yes": 1, "No": 0})
)

df_fe["book_table"] = (
    df_fe["book_table"]
    .map({"Yes": 1, "No": 0})
)

In [58]:
df_fe["log_votes"] = np.log1p(df_fe["votes"])

print(
    df_fe[["votes", "log_votes"]].describe()
)

              votes     log_votes
count  51696.000000  51696.000000
mean     283.812771      3.572843
std      803.981766      2.315259
min        0.000000      0.000000
25%        7.000000      2.079442
50%       41.000000      3.737670
75%      198.000000      5.293305
max    16832.000000      9.731097


In [59]:
df_fe["log_cost"] = np.log1p(
    df_fe["approx_costfor_two_people"]
)

In [60]:
df_fe["cost_band"] = pd.cut(
    df_fe["approx_costfor_two_people"],
    bins=[0, 300, 600, 1000, 2000, np.inf],
    labels=[
        "Budget",
        "Mid-Range",
        "Upper-Mid",
        "Premium",
        "Luxury"
    ]
)

In [61]:
df_fe["cuisine_count"] = (
    df_fe["cuisines"]
    .str.split(",")
    .str.len()
)

In [62]:
df_fe["primary_cuisine"] = (
    df_fe["cuisines"]
    .str.split(",")
    .str[0]
    .str.strip()
)

In [63]:
print(df_fe[[
    "cuisines",
    "cuisine_count",
    "primary_cuisine"
]].head(15))

                                       cuisines  cuisine_count primary_cuisine
0                North Indian, Mughlai, Chinese              3    North Indian
1                   Chinese, North Indian, Thai              3         Chinese
2                        Cafe, Mexican, Italian              3            Cafe
3                    South Indian, North Indian              2    South Indian
4                      North Indian, Rajasthani              2    North Indian
5                                  North Indian              1    North Indian
6   North Indian, South Indian, Andhra, Chinese              4    North Indian
7                          Pizza, Cafe, Italian              3           Pizza
8                    Cafe, Italian, Continental              3            Cafe
9      Cafe, Mexican, Italian, Momos, Beverages              5            Cafe
10                                         Cafe              1            Cafe
11                   Cafe, Italian, Continental     

In [64]:
df_fe["primary_rest_type"] = (
    df_fe["rest_type"]
    .str.split(",")
    .str[0]
    .str.strip()
)

Keep only restaurants that have a rating
Our target is based partly on rating, so restaurants without ratings cannot have a historical performance score.

In [65]:
df_target = df_fe[df_fe["rate"].notna()].copy()

In [66]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_target,
    test_size=0.20,
    random_state=42
)

In [67]:
train_df["rating_percentile"] = (
    train_df["rate"]
    .rank(pct=True)
)

train_df["engagement_percentile"] = (
    train_df["log_votes"]
    .rank(pct=True)
)

train_df["performance_score"] = (
    0.5 * train_df["rating_percentile"]
    + 0.5 * train_df["engagement_percentile"]
)

In [68]:
low_threshold = train_df["performance_score"].quantile(0.33)
high_threshold = train_df["performance_score"].quantile(0.67)

print("Low threshold:", low_threshold)
print("High threshold:", high_threshold)

Low threshold: 0.3364034561382455
High threshold: 0.6507035281411256


In [69]:
train_df["performance_class"] = pd.cut(
    train_df["performance_score"],
    bins=[-np.inf, low_threshold, high_threshold, np.inf],
    labels=["Low", "Medium", "High"]
)

 Apply Training-Derived Target Rules to Test Data

In [70]:
train_final = train_df.copy()
test_final = test_df.copy()

In [71]:
train_rate_values = train_final["rate"].sort_values().to_numpy()
train_vote_values = train_final["log_votes"].sort_values().to_numpy()

In [72]:
test_final["rating_percentile"] = (
    np.searchsorted(
        train_rate_values,
        test_final["rate"],
        side="right"
    ) / len(train_rate_values)
)

test_final["engagement_percentile"] = (
    np.searchsorted(
        train_vote_values,
        test_final["log_votes"],
        side="right"
    ) / len(train_vote_values)
)

In [73]:
test_final["performance_score"] = (
    0.5 * test_final["rating_percentile"]
    +
    0.5 * test_final["engagement_percentile"]
)

In [74]:
test_final["performance_class"] = pd.cut(
    test_final["performance_score"],
    bins=[-np.inf, low_threshold, high_threshold, np.inf],
    labels=["Low", "Medium", "High"]
)

In [75]:
location_counts = (
    train_final["location"]
    .value_counts()
    .rename("historical_restaurant_count")
    .reset_index()
)

location_counts.columns = [
    "location",
    "historical_restaurant_count"
]

display(location_counts.head(10))

,location,historical_restaurant_count
0,BTM,3149
1,Koramangala 5th Block,1893
2,HSR,1630
3,Indiranagar,1461
4,JP Nagar,1374
5,Jayanagar,1315
6,Whitefield,1274
7,Marathahalli,1149
8,Bannerghatta Road,980
9,Koramangala 7th Block,871


In [76]:
train_final = train_final.merge(
    location_counts,
    on="location",
    how="left"
)

In [77]:
test_final = test_final.merge(
    location_counts,
    on="location",
    how="left"
)

test_final["historical_restaurant_count"] = (
    test_final["historical_restaurant_count"]
    .fillna(0)
)

In [78]:
location_cost = (
    train_final.groupby("location")["approx_costfor_two_people"]
    .median()
    .rename("location_median_cost")
    .reset_index()
)

display(location_cost.head(10))

,location,location_median_cost
0,BTM,400.0
1,Banashankari,400.0
2,Banaswadi,400.0
3,Bannerghatta Road,400.0
4,Basavanagudi,300.0
5,Basaveshwara Nagar,400.0
6,Bellandur,500.0
7,Bommanahalli,400.0
8,Brigade Road,500.0
9,Brookefield,400.0


In [79]:
train_final = train_final.merge(
    location_cost,
    on="location",
    how="left"
)

In [80]:
test_final = test_final.merge(
    location_cost,
    on="location",
    how="left"
)

In [81]:
train_cost_median = train_final[
    "approx_costfor_two_people"
].median()

test_final["location_median_cost"] = (
    test_final["location_median_cost"]
    .fillna(train_cost_median)
)

In [82]:
location_service = (
    train_final.groupby("location")
    .agg(
        location_online_order_rate=("online_order", "mean"),
        location_book_table_rate=("book_table", "mean")
    )
    .reset_index()
)


In [83]:
train_final = train_final.merge(
    location_service,
    on="location",
    how="left"
)

In [84]:
test_final = test_final.merge(
    location_service,
    on="location",
    how="left"
)

In [85]:
overall_online_rate = train_final["online_order"].mean()
overall_booking_rate = train_final["book_table"].mean()

test_final["location_online_order_rate"] = (
    test_final["location_online_order_rate"]
    .fillna(overall_online_rate)
)

test_final["location_book_table_rate"] = (
    test_final["location_book_table_rate"]
    .fillna(overall_booking_rate)
)

In [86]:
location_diversity = (
    train_final.groupby("location")
    .agg(
        location_cuisine_diversity=("primary_cuisine", "nunique"),
        location_business_type_diversity=("primary_rest_type", "nunique")
    )
    .reset_index()
)

display(location_diversity.head(10))

,location,location_cuisine_diversity,location_business_type_diversity
0,BTM,38,14
1,Banashankari,33,13
2,Banaswadi,27,14
3,Bannerghatta Road,33,15
4,Basavanagudi,26,11
5,Basaveshwara Nagar,18,10
6,Bellandur,42,16
7,Bommanahalli,15,9
8,Brigade Road,31,14
9,Brookefield,28,13


In [87]:
train_final = train_final.merge(
    location_diversity,
    on="location",
    how="left"
)

In [88]:
test_final = test_final.merge(
    location_diversity,
    on="location",
    how="left"
)

In [89]:
train_cuisine_diversity = (
    train_final["primary_cuisine"].nunique()
)

train_business_diversity = (
    train_final["primary_rest_type"].nunique()
)

test_final["location_cuisine_diversity"] = (
    test_final["location_cuisine_diversity"]
    .fillna(train_cuisine_diversity)
)

test_final["location_business_type_diversity"] = (
    test_final["location_business_type_diversity"]
    .fillna(train_business_diversity)
)

In [90]:
feature_audit = pd.DataFrame({
    "feature": [
        "rate",
        "votes",
        "log_votes",
        "rating_percentile",
        "engagement_percentile",
        "performance_score",
        "performance_class",
        "online_order",
        "book_table",
        "approx_costfor_two_people",
        "log_cost",
        "cost_band",
        "location",
        "primary_cuisine",
        "cuisine_count",
        "primary_rest_type",
        "historical_restaurant_count",
        "location_median_cost",
        "location_online_order_rate",
        "location_book_table_rate",
        "location_cuisine_diversity",
        "location_business_type_diversity"
    ],
    "role": [
        "Target component",
        "Target component",
        "Target component",
        "Target construction",
        "Target construction",
        "Target",
        "Target",
        "Business input",
        "Business input",
        "Business input",
        "Business input",
        "Business input",
        "Locality input",
        "Business input",
        "Business input",
        "Business input",
        "Locality input",
        "Locality input",
        "Locality input",
        "Locality input",
        "Locality input",
        "Locality input"
    ],
    "use_as_model_input": [
        "No",
        "No",
        "No",
        "No",
        "No",
        "No",
        "No",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Test",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ]
})

display(feature_audit)

,feature,role,use_as_model_input
0,rate,Target component,No
1,votes,Target component,No
2,log_votes,Target component,No
3,rating_percentile,Target construction,No
4,engagement_percentile,Target construction,No
5,performance_score,Target,No
6,performance_class,Target,No
7,online_order,Business input,Yes
8,book_table,Business input,Yes
9,approx_costfor_two_people,Business input,Yes


In [91]:
feature_columns = [
    "online_order",
    "book_table",
    "approx_costfor_two_people",
    "log_cost",
    "cost_band",
    "location",
    "primary_cuisine",
    "cuisine_count",
    "primary_rest_type",
    "historical_restaurant_count",
    "location_median_cost",
    "location_online_order_rate",
    "location_book_table_rate",
    "location_cuisine_diversity",
    "location_business_type_diversity"
]

X_train = train_final[feature_columns].copy()
X_test = test_final[feature_columns].copy()

y_train = train_final["performance_class"].copy()
y_test = test_final["performance_class"].copy()

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (33332, 15)
X_test : (8333, 15)
y_train: (33332,)
y_test : (8333,)


In [92]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [93]:
categorical_features = [
    "online_order",
    "book_table",
    "cost_band",
    "location",
    "primary_cuisine",
    "primary_rest_type"
]

numerical_features = [
    "approx_costfor_two_people",
    "log_cost",
    "cuisine_count",
    "historical_restaurant_count",
    "location_median_cost",
    "location_online_order_rate",
    "location_book_table_rate",
    "location_cuisine_diversity",
    "location_business_type_diversity"
]

In [94]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

In [95]:
X_train_encoded = preprocessor.fit_transform(X_train)

X_test_encoded = preprocessor.transform(X_test)

In [96]:
import joblib
joblib.dump(
    preprocessor,
    "../models/zomato_preprocessor.pkl"
)

print("Preprocessor saved successfully.")

Preprocessor saved successfully.


In [97]:
import os

os.makedirs("../data/processed", exist_ok=True)

train_final.to_csv(
    "../data/processed/zomato_train_engineered.csv",
    index=False
)

test_final.to_csv(
    "../data/processed/zomato_test_engineered.csv",
    index=False
)

print("Train and test engineered datasets saved.")

Train and test engineered datasets saved.
